# Solution A1 · Three scalers from scratch: fit, transform and the constant column case

**Type**: Implement from scratch

## Objective

Implement `StandardScaler`, `MinMaxScaler` and `RobustScaler` with the same interface as scikit-learn (`fit`,
`transform`, `fit_transform`, `inverse_transform`), check them against the original versions and handle the edge case
of a constant column.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

df = ds.loans(with_missing=False)
X_num = df[["age", "income", "job_tenure", "amount"]].to_numpy()
X_num = np.column_stack([X_num, np.full(len(X_num), 7.0)])         # a constant column: the edge case
X_tr, X_te = train_test_split(X_num, test_size=0.25, random_state=0)

## Commented solution

In [ ]:
def _safe_scale(s):
    return np.where(s == 0, 1.0, s)                 # constant column: dividing by 1 leaves zeros, not NaN


class MyStandardScaler:
    def fit(self, X):
        self.mean_ = X.mean(axis=0)
        self.scale_ = _safe_scale(X.std(axis=0))
        return self

    def transform(self, X):
        return (X - self.mean_) / self.scale_

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def inverse_transform(self, Z):
        return Z * self.scale_ + self.mean_


class MyMinMaxScaler:
    def fit(self, X):
        self.min_ = X.min(axis=0)
        self.scale_ = _safe_scale(X.max(axis=0) - self.min_)
        return self

    def transform(self, X):
        return (X - self.min_) / self.scale_

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def inverse_transform(self, Z):
        return Z * self.scale_ + self.min_


class MyRobustScaler:
    def fit(self, X):
        self.center_ = np.median(X, axis=0)
        q25, q75 = np.percentile(X, [25, 75], axis=0)
        self.scale_ = _safe_scale(q75 - q25)
        return self

    def transform(self, X):
        return (X - self.center_) / self.scale_

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def inverse_transform(self, Z):
        return Z * self.scale_ + self.center_

In [ ]:
scaler_tr = MyStandardScaler().fit(X_tr)
refit_difference = np.abs(MyStandardScaler().fit_transform(X_te) - scaler_tr.transform(X_te)).max()
print("maximum difference between wrong and right code on the test set:", round(refit_difference, 3))
print("means of the correctly scaled test set:", scaler_tr.transform(X_te).mean(axis=0).round(3))

## Check

In [ ]:
with verify():
    pairs = [(MyStandardScaler, StandardScaler), (MyMinMaxScaler, MinMaxScaler), (MyRobustScaler, RobustScaler)]
    for Mine, Ref in pairs:
        mine, ref = Mine(), Ref()
        check(close(mine.fit_transform(X_tr), ref.fit_transform(X_tr), atol=1e-9),
              f"{Mine.__name__}: fit_transform matches scikit-learn on the training set",
              f"{Mine.__name__}.fit_transform(X_tr) does not match {Ref.__name__}")
        check(close(mine.transform(X_te), ref.transform(X_te), atol=1e-9),
              f"{Mine.__name__}: transform uses the training statistics",
              f"{Mine.__name__}.transform(X_te) does not match: did you recompute the statistics on the test set?")
        check(close(mine.inverse_transform(mine.transform(X_te)), X_te, atol=1e-6),
              f"{Mine.__name__}: inverse_transform rebuilds the data", "transform followed by inverse_transform must give back X")
        check(not np.isnan(mine.transform(X_te)).any() and np.all(mine.transform(X_te)[:, 4] == 0),
              f"{Mine.__name__}: the constant column becomes zero, with no NaN", "where the scale is 0 use 1")
    s = MyStandardScaler().fit(X_tr)
    expected = np.abs(MyStandardScaler().fit_transform(X_te) - s.transform(X_te)).max()
    check(close(refit_difference, expected) and refit_difference > 0.1,
          "refitting the scaler on the test set changes the data substantially",
          "refit_difference = max |fit_transform(X_te) - scaler_tr.transform(X_te)|")
done()

## What you learned

- A scaler is an object with state: `fit` computes some numbers on the training set, `transform` uses them. Recomputing them on the test set
  (`fit_transform(X_te)`) shifts and rescales the test differently from the training: the model sees data that do not
  resemble those it was trained on.
- A constant column has standard deviation zero: without the `scale == 0 → 1` protection the division produces NaN and
  the NaN spreads through the whole model. scikit-learn has the same protection (`_handle_zeros_in_scale`).
- The three scalers differ only in the center (mean, minimum, median) and the scale (standard deviation, total
  range, interquartile range): everything else is identical.

## Common pitfall

Computing the statistics on the complete dataset "for convenience" before the split. The model never sees the test set, but
the scaler does: the statistic of the test set enters the training data. The effect on the metrics is small with scaling and
large with encoders that use the target (exercise A3).

## Interview question

> *Click on a question to show its answer.*

<details><summary>In production your service receives one request at a time. How do you apply the scaler to a single row?</summary>

With `transform`, using the mean and standard deviation saved from training: that is why the scaler must be serialized together with the
model (better the whole Pipeline). With `fit_transform` on one row the standard deviation would be zero and the result
would be meaningless.

</details>